In [ ]:
from pathlib import Path

base_path = Path("../data/interim/")

region_files = {
    "Ariana" : base_path / "poverty_ariana_2015.csv",
    "Beja" : base_path / "poverty_beja_2015.csv",
    "Ben Arous" : base_path / "poverty_ben_arous_2015.csv",
    "Bizerte" : base_path / "poverty_bizerte_2015.csv",
    "Gabes" : base_path / "poverty_gabes_2015.csv",
    "Gafsa" : base_path / "poverty_gafsa_2015.csv",
    "Jendouba" : base_path / "poverty_jendouba_2015.csv",
    "Kairouan" : base_path / "poverty_kairouan_2015.csv",
    "Kasserine" : base_path / "poverty_kasserine_2015.csv",
    "Kebili" : base_path / "poverty_kebili_2015.csv",
    "Kef" : base_path / "poverty_kef_2015.csv",
    "Mahdia" : base_path / "poverty_mahdia_2015.csv",
    "Manouba" : base_path / "poverty_manouba_2015.csv",
    "Medenine" : base_path / "poverty_medenine_2015.csv",
    "Monastir" : base_path / "poverty_monastir_2015.csv",
    "Nabeul" : base_path / "poverty_nabeul_2015.csv",
    "Sfax" : base_path / "poverty_sfax_2015.csv",
    "Sidi Bouzid" : base_path / "poverty_sidi_bouzid_2015.csv",
    "Siliana" : base_path / "poverty_siliana_2015.csv",
    "Sousse" : base_path / "poverty_sousse_2015.csv",
    "Tataouine" : base_path / "poverty_tataouine_2015.csv",
    "Tozeur" : base_path / "poverty_tozeur_2015.csv",
    "Tunis" : base_path / "poverty_tunis_2015.csv",
    "Zaghouan" : base_path / "poverty_zaghouan_2015.csv",
}


In [ ]:
def load_region_data(region_name):
    file_path = region_files.get(region_name)
    if not file_path or not file_path.exists():
        raise ValueError(f"Data for region '{region_name}' not found.")
    
    with open(file_path, 'r', encoding='utf-8') as f:
        data = f.readlines()
    
    return data

In [ ]:
def clean_region_data(df, governorate_name, strict=True):
    col0 = df.iloc[:, 0].astype(str)

    # delegation_mask = col0.str.match(r'[A-Z]+(?: [A-Z]+)*\b', na=False)
    delegation_mask = col0.str.match(r'^[A-ZÀ-Ÿ]+(?: [A-ZÀ-Ÿ]+)*$', na=False)

    not_header_mask = ~col0.str.contains(r'(?i)source|d[eé]l[eé]gation|tableau|carte', na=False)

    filtered_df = df[delegation_mask & not_header_mask].copy()

    col0_filtered = filtered_df.iloc[:, 0].astype(str)
    validation_mask = col0_filtered.str.match(r'^[A-Z]+(?: [A-Z]+)*\b', na=False)

    if not validation_mask.all():
        msg = (
            f"[Warning] Some rows do not satisfy the delegation rule after cleaning "
            f"in governorate '{governorate_name}'.\n"
            f"Invalid rows:\n{filtered_df[~validation_mask]}"
        )
        if strict:
            raise ValueError(msg)
        else:
            print(msg)


    filtered_df.iloc[:, 1:] = (
        filtered_df.iloc[:, 1:]
        .replace(",", ".", regex=True)
        .astype(float)
    )

    filtered_df.columns = ["Delegation", "PrimDrop%", "SecDrop%", "TotalDrop%", "Poverty%"]
    
    filtered_df["Delegation"] = (
        filtered_df["Delegation"]
        .str.title()
        .str.replace("Sud", "South")
        .str.replace("Nord", "North")
        .str.replace("Ouest", "West")
        .str.replace("Est", "East")
        .str.replace("Ville", "City")
        .str.replace("Nouvelle", "New")
        .str.replace("Superieur", "Superior")
    )

    filtered_df["Governorate"] = governorate_name
    filtered_df.reset_index(drop=True, inplace=True)
    return filtered_df


,Delegation,PrimDrop%,SecDrop%,TotalDrop%,Poverty%,Governorate
0,Ennadhour,0.5,7.4,3.5,22.5,Zaghouan
1,Saouaf,0.2,9.0,3.4,19.5,Zaghouan
2,Bir Mcharga,0.2,9.4,3.6,15.8,Zaghouan
3,Zriba,0.3,5.2,2.2,13.0,Zaghouan
4,El Fahs,0.2,5.4,2.5,12.5,Zaghouan
5,Zaghouan,0.3,7.4,3.7,7.2,Zaghouan


In [105]:
# Now we will store all of this in a single dataframe and store it as a CSV
import pandas as pd

tunisia_districts = {
    1: ["Bizerte", "Beja", "Jendouba", "Kef"],
    2: ["Tunis", "Ariana", "Ben Arous", "Zaghouan", "Manouba", "Nabeul"],
    3: ["Siliana", "Sousse", "Kasserine", "Kairouan", "Monastir", "Mahdia"],
    4: ["Tozeur", "Sidi Bouzid", "Sfax", "Gafsa"],
    5: ["Tataouine", "Gabes", "Kebili", "Medenine"],
}

all_data = []
i = 1
for district_num, governorates in tunisia_districts.items():
    for governorate in governorates:
        print(f"Processing the {i}th governorate: {governorate.upper()}...")
        raw_data = load_region_data(governorate)
        df_cleaned = clean_region_data(raw_data, governorate, strict=False)
        df_cleaned["District"] = district_num
        all_data.append(df_cleaned)
        i += 1

final_df = pd.concat(all_data, ignore_index=True)
final_df.to_csv("../data/clean/tunisia_poverty_2015_cleaned.csv", index=False)

Processing the 1th governorate: BIZERTE...
Processing the 2th governorate: BEJA...
Processing the 3th governorate: JENDOUBA...
Processing the 4th governorate: KEF...
Processing the 5th governorate: TUNIS...
Processing the 6th governorate: ARIANA...
Processing the 7th governorate: BEN AROUS...
Processing the 8th governorate: ZAGHOUAN...
Processing the 9th governorate: MANOUBA...
Processing the 10th governorate: NABEUL...
Processing the 11th governorate: SILIANA...
Processing the 12th governorate: SOUSSE...
Processing the 13th governorate: KASSERINE...
Processing the 14th governorate: KAIROUAN...
Processing the 15th governorate: MONASTIR...
Processing the 16th governorate: MAHDIA...
Processing the 17th governorate: TOZEUR...
Processing the 18th governorate: SIDI BOUZID...
Processing the 19th governorate: SFAX...
Processing the 20th governorate: GAFSA...
Processing the 21th governorate: TATAOUINE...
Processing the 22th governorate: GABES...
Processing the 23th governorate: KEBILI...
Proce